<a href="https://colab.research.google.com/github/balasri03/Mini_project/blob/main/vast_bert_bart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dharanimanchala_vasty123_path = kagglehub.dataset_download('dharanimanchala/vasty123')

print('Data source import complete.')


In [ ]:
!pip install transformers datasets -q

In [ ]:
!pip install seaborn matplotlib -q

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import re
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    BartTokenizer,
    BartForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from tqdm import tqdm

In [ ]:
train_df = pd.read_csv("/kaggle/input/datasets/dharanimanchala/vasty123/VAST_train_11k.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/dharanimanchala/vasty123/VAST_val.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/dharanimanchala/vasty123/VAST_test.csv")

print(train_df.shape)
train_df.head()

(11305, 4)


,Tweet,Target 1,Stance 1,seen?
0,Regulation of corporations has been subverted ...,company,AGAINST,1
1,Regulation of corporations has been subverted ...,regulation,AGAINST,1
2,Regulation of corporations has been subverted ...,corporate regulation,AGAINST,1
3,Regulation of corporations has been subverted ...,regulation corporation,AGAINST,1
4,Absolutely it's needs to be defined and regula...,food label,AGAINST,1


In [ ]:
train_df = train_df[['Tweet','Target 1','Stance 1']].dropna()
val_df   = val_df[['Tweet','Target 1','Stance 1']].dropna()
test_df  = test_df[['Tweet','Target 1','Stance 1']].dropna()

train_df['text'] = train_df['Target 1'] + " [SEP] " + train_df['Tweet']
val_df['text']   = val_df['Target 1'] + " [SEP] " + val_df['Tweet']
test_df['text']  = test_df['Target 1'] + " [SEP] " + test_df['Tweet']

train_df = train_df[['text','Stance 1']]
val_df   = val_df[['text','Stance 1']]
test_df  = test_df[['text','Stance 1']]

train_df.columns = ['text','label']
val_df.columns   = ['text','label']
test_df.columns  = ['text','label']

In [ ]:
def clean_text(text):

    text = text.lower()                         # lowercase
    text = re.sub(r"http\S+"," ",text)           # remove URLs
    text = re.sub(r"@\w+"," ",text)              # remove mentions
    text = re.sub(r"#"," ",text)                 # remove hashtags
    text = re.sub(r"\d+"," ",text)               # remove numbers
    text = re.sub(r"[^\w\s]"," ",text)           # remove punctuation
    text = re.sub(r"\s+"," ",text).strip()       # remove extra spaces

    return text

In [ ]:
train_df['text'] = train_df['text'].apply(clean_text)
val_df['text']   = val_df['text'].apply(clean_text)
test_df['text']  = test_df['text'].apply(clean_text)

In [ ]:
le = LabelEncoder()

train_df['label'] = le.fit_transform(train_df['label'])
val_df['label']   = le.transform(val_df['label'])
test_df['label']  = le.transform(test_df['label'])

num_labels = len(le.classes_)
print("Classes:", le.classes_)

Classes: ['AGAINST' 'FAVOR' 'NONE']


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
class StanceDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
BATCH_SIZE = 16

train_dataset = StanceDataset(
    train_df.text.values,
    train_df.label.values,
    tokenizer,
    MAX_LEN
)

val_dataset = StanceDataset(
    val_df.text.values,
    val_df.label.values,
    tokenizer,
    MAX_LEN
)

test_dataset = StanceDataset(
    test_df.text.values,
    test_df.label.values,
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
EPOCHS = 4
LR = 2e-5

optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1*total_steps),
    num_training_steps=total_steps
)

In [ ]:
def train_epoch(model, loader):

    model.train()
    losses = []

    for batch in tqdm(loader):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention,
            labels=labels
        )

        loss = outputs.loss
        losses.append(loss.item())

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        optimizer.step()
        scheduler.step()

    return np.mean(losses)

In [ ]:
def evaluate(model, loader):

    model.eval()

    preds = []
    true = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch['input_ids'].to(device)
            attention = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention
            )

            logits = outputs.logits

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true.extend(labels.cpu().numpy())

    acc = accuracy_score(true, preds)
    prec = precision_score(true, preds, average='weighted', zero_division=0)
    rec = recall_score(true, preds, average='weighted')
    f1 = f1_score(true, preds, average='weighted')

    return acc, prec, rec, f1

In [ ]:
start = time.time()

for epoch in range(EPOCHS):

    train_loss = train_epoch(model,train_loader)

    val_acc,val_prec,val_rec,val_f1 = evaluate(model,val_loader)

    print(f"\nEpoch {epoch+1}")
    print("Train Loss:",train_loss)
    print("Val Accuracy:",val_acc)
    print("Val F1:",val_f1)

train_time = (time.time()-start)/3600

100%|██████████| 707/707 [04:23<00:00,  2.68it/s]



Epoch 1
Train Loss: 0.8633656716751411
Val Accuracy: 0.6692531522793405
Val F1: 0.6588730778421152


100%|██████████| 707/707 [04:30<00:00,  2.61it/s]



Epoch 2
Train Loss: 0.5935968763841767
Val Accuracy: 0.6789524733268671
Val F1: 0.6632970603161616


100%|██████████| 707/707 [04:31<00:00,  2.61it/s]



Epoch 3
Train Loss: 0.4411896032609225
Val Accuracy: 0.6644034917555771
Val F1: 0.6303648720908399


100%|██████████| 707/707 [04:31<00:00,  2.61it/s]



Epoch 4
Train Loss: 0.3468119282923544
Val Accuracy: 0.6998060135790495
Val F1: 0.6919062201282047


In [ ]:
start = time.time()

acc,prec,rec,f1 = evaluate(model,test_loader)

infer_time = time.time()-start

print("\nBERT Results")
print("Accuracy:",acc)
print("Precision:",prec)
print("Recall:",rec)
print("F1:",f1)
print("Train Time(h):",train_time)
print("Infer Time(s):",infer_time)


BERT Results
Accuracy: 0.6949434464404525
Precision: 0.7016707349877939
Recall: 0.6949434464404525
F1: 0.6956959033467612
Train Time(h): 0.3180946100420422
Infer Time(s): 24.712908029556274


In [ ]:
total_params = sum(p.numel() for p in model.parameters())

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("BERT Model Parameters")
print("----------------------")
print(f"Total Parameters      : {total_params}")
print(f"Trainable Parameters  : {trainable_params}")
print(f"Parameters (Millions) : {total_params/1e6:.2f}M")

BERT Model Parameters
----------------------
Total Parameters      : 140013315
Trainable Parameters  : 140013315
Parameters (Millions) : 140.01M


In [ ]:
from transformers import BartModel
import torch.nn as nn

class BartClassifier(nn.Module):
    def __init__(self, num_labels):
        super(BartClassifier, self).__init__()
        self.bart = BartModel.from_pretrained("facebook/bart-base")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bart.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bart(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 🔥 take CLS-like token (first token)
        pooled_output = outputs.last_hidden_state[:, 0, :]

        x = self.dropout(pooled_output)
        logits = self.fc(x)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
NUM_CLASSES = 3   # your dataset

model = BartClassifier(NUM_CLASSES)
model.to(device)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

BartClassifier(
  (bart): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm

In [ ]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

In [ ]:
class StanceDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
def train_epoch(model, loader):
    model.train()
    losses = []

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention,
            labels=labels
        )

        loss = outputs["loss"]
        losses.append(loss.item())

        loss.backward()
        optimizer.step()
        scheduler.step()

    return np.mean(losses)

In [ ]:
def evaluate(model, loader):
    model.eval()

    preds = []
    true = []

    with torch.no_grad():
        for batch in loader:

            input_ids = batch['input_ids'].to(device)
            attention = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention
            )

            logits = outputs["logits"]

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true.extend(labels.cpu().numpy())

    acc = accuracy_score(true, preds)
    prec = precision_score(true, preds, average='weighted', zero_division=0)
    rec = recall_score(true, preds, average='weighted')
    f1 = f1_score(true, preds, average='weighted')

    return acc, prec, rec, f1

In [ ]:
BATCH_SIZE = 16

train_dataset = StanceDataset(
    train_df.text.values,
    train_df.label.values,
    tokenizer,
    MAX_LEN
)

val_dataset = StanceDataset(
    val_df.text.values,
    val_df.label.values,
    tokenizer,
    MAX_LEN
)

test_dataset = StanceDataset(
    test_df.text.values,
    test_df.label.values,
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
EPOCHS = 4
LR = 2e-5

optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)



In [ ]:
def train_epoch(model, loader):
    model.train()
    losses = []

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention,
            labels=labels
        )

        loss = outputs["loss"]
        losses.append(loss.item())

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    return np.mean(losses)

In [ ]:
def evaluate(model, loader):
    model.eval()

    preds = []
    true = []

    with torch.no_grad():
        for batch in loader:

            input_ids = batch['input_ids'].to(device)
            attention = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention
            )

            logits = outputs["logits"]

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true.extend(labels.cpu().numpy())

    acc = accuracy_score(true, preds)
    prec = precision_score(true, preds, average='weighted', zero_division=0)
    rec = recall_score(true, preds, average='weighted')
    f1 = f1_score(true, preds, average='weighted')

    return acc, prec, rec, f1

In [ ]:
start = time.time()

for epoch in range(EPOCHS):

    train_loss = train_epoch(model, train_loader)

    val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Accuracy:", val_acc)
    print("Val F1:", val_f1)

train_time = (time.time() - start) / 3600


Epoch 1
Train Loss: 1.106475896216518
Val Accuracy: 0.6779825412221144
Val F1: 0.6667453236544973

Epoch 2
Train Loss: 0.7473875548074067
Val Accuracy: 0.6663433559650824
Val F1: 0.6580110172840645

Epoch 3
Train Loss: 0.6095143461539351
Val Accuracy: 0.7075654704170709
Val F1: 0.7105204776917357

Epoch 4
Train Loss: 0.525451608876865
Val Accuracy: 0.7085354025218235
Val F1: 0.7092376025123635


In [ ]:
start = time.time()

acc, prec, rec, f1 = evaluate(model, test_loader)

infer_time = time.time() - start

print("\nBART Results")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print("Train Time(h):", train_time)
print("Infer Time(s):", infer_time)


BART Results
Accuracy: 0.709913506320692
Precision: 0.7095116344920676
Recall: 0.709913506320692
F1: 0.7096067205170199
Train Time(h): 0.3803009494145711
Infer Time(s): 28.789350986480713


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nBART Model Parameters")
print("----------------------")
print("Total Parameters      :", total_params)
print("Trainable Parameters  :", trainable_params)
print("Parameters (Millions) :", total_params / 1e6, "M")


BART Model Parameters
----------------------
Total Parameters      : 139422723
Trainable Parameters  : 139422723
Parameters (Millions) : 139.422723 M
